# Feature Dataset for Building Power usage Dataset

In [10]:
import pandas as pd 

index = pd.date_range(start="2016-01-01 00:00:00", end="2020-09-30 23:59:45", freq="15min")
Feature_dataset = pd.DataFrame(index = index)

Feature_dataset



""
2016-01-01 00:00:00
2016-01-01 00:15:00
2016-01-01 00:30:00
2016-01-01 00:45:00
2016-01-01 01:00:00
...
2020-09-30 22:45:00
2020-09-30 23:00:00
2020-09-30 23:15:00
2020-09-30 23:30:00


In [11]:
import pandas as pd
import numpy as np

# ============================================================
# 0. PREPARE DATETIME INDEX
# ============================================================

Feature_dataset.index = pd.to_datetime(Feature_dataset.index)
Feature_dataset = Feature_dataset.sort_index()

date = Feature_dataset.index.normalize()
hour = Feature_dataset.index.hour
minute = Feature_dataset.index.minute
time_minutes = hour * 60 + minute


# ============================================================
# 1. HELPER FUNCTION FOR DATE RANGES
# ============================================================

def in_date_ranges(index, ranges):
    dates = index.normalize()
    mask = pd.Series(False, index=index)

    for start, end in ranges:
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)

        mask |= (dates >= start) & (dates <= end)

    return mask.astype(int).values


# ============================================================
# 2. WEEKDAY / WEEKEND
# ============================================================

Feature_dataset["is_weekday"] = (
    Feature_dataset.index.dayofweek < 5
).astype(int)

Feature_dataset["is_weekend"] = (
    Feature_dataset.index.dayofweek >= 5
).astype(int)


# ============================================================
# 3. UNIVERSITY OPEN / CLOSED
# ============================================================

university_open_ranges = [
    ("2016-01-04", "2016-12-22"),
    ("2017-01-03", "2017-12-21"),
    ("2018-01-02", "2018-12-21"),
    ("2019-01-02", "2019-12-23"),
    ("2020-01-02", "2020-12-23"),
]

Feature_dataset["university_open"] = in_date_ranges(
    Feature_dataset.index,
    university_open_ranges
)

Feature_dataset["university_closed"] = (
    1 - Feature_dataset["university_open"]
)


# ============================================================
# 4. CAMPUS FACILITY HOURS
# Mon-Fri: 09:00 - 17:00
# ============================================================

campus_time_open = (
    (time_minutes >= 9 * 60) &
    (time_minutes < 17 * 60)
)

Feature_dataset["campus_facility_hours"] = (
    (Feature_dataset["is_weekday"] == 1) &
    campus_time_open
).astype(int)


# ============================================================
# 5. LIBRARY OPENING HOURS
#
# Weekdays: 08:00 - 22:00
# Weekends: 10:00 - 17:00
# ============================================================

weekday_library = (
    (Feature_dataset["is_weekday"] == 1) &
    (time_minutes >= 8 * 60) &
    (time_minutes < 22 * 60)
)

weekend_library = (
    (Feature_dataset["is_weekend"] == 1) &
    (time_minutes >= 10 * 60) &
    (time_minutes < 17 * 60)
)

Feature_dataset["library_open_hours"] = (
    weekday_library | weekend_library
).astype(int)


# ============================================================
# 6. SEMESTER 1
# ============================================================

semester_1_ranges = [
    ("2016-02-29", "2016-05-27"),
    ("2017-02-21", "2017-05-26"),
    ("2018-02-26", "2018-05-25"),
    ("2019-03-04", "2019-05-31"),
    ("2020-03-09", "2020-06-12"),
]

Feature_dataset["semester_1"] = in_date_ranges(
    Feature_dataset.index,
    semester_1_ranges
)


# ============================================================
# 7. SEMESTER 2
# ============================================================

semester_2_ranges = [
    ("2016-07-25", "2016-10-21"),
    ("2017-07-24", "2017-10-20"),
    ("2018-07-23", "2018-10-19"),
    ("2019-07-29", "2019-10-25"),
    ("2020-08-03", "2020-11-06"),
]

Feature_dataset["semester_2"] = in_date_ranges(
    Feature_dataset.index,
    semester_2_ranges
)

Feature_dataset["semester_active"] = (
    (Feature_dataset["semester_1"] == 1) |
    (Feature_dataset["semester_2"] == 1)
).astype(int)


# ============================================================
# 8. MID-SEMESTER BREAK
# ============================================================

mid_sem_break_ranges = [
    ("2016-03-25", "2016-04-01"),
    ("2016-09-26", "2016-09-30"),

    ("2017-04-14", "2017-04-21"),
    ("2017-09-25", "2017-09-29"),

    ("2018-03-30", "2018-04-06"),
    ("2018-09-24", "2018-09-28"),

    ("2019-04-19", "2019-04-26"),
    ("2019-09-30", "2019-10-04"),

    ("2020-04-10", "2020-04-17"),
    ("2020-09-21", "2020-10-02"),
]

Feature_dataset["mid_sem_break"] = in_date_ranges(
    Feature_dataset.index,
    mid_sem_break_ranges
)


# ============================================================
# 9. ORIENTATION WEEK
# ============================================================

orientation_ranges = [
    ("2016-02-22", "2016-02-26"),
    ("2016-07-18", "2016-07-22"),

    ("2017-02-20", "2017-02-24"),
    ("2017-07-17", "2017-07-21"),

    ("2018-02-19", "2018-02-23"),
    ("2018-07-16", "2018-07-20"),

    ("2019-02-25", "2019-03-01"),
    ("2019-07-22", "2019-07-26"),

    # Semester 1 2020 not listed
    ("2020-07-27", "2020-07-31"),
]

Feature_dataset["orientation_week"] = in_date_ranges(
    Feature_dataset.index,
    orientation_ranges
)


# ============================================================
# 10. SWOT VAC
# ============================================================

swotvac_ranges = [
    ("2016-05-30", "2016-06-03"),
    ("2016-10-24", "2016-10-28"),

    ("2017-05-29", "2017-06-02"),
    ("2017-10-23", "2017-10-27"),

    ("2018-05-28", "2018-06-01"),
    ("2018-10-22", "2018-10-26"),

    ("2019-06-03", "2019-06-07"),
    ("2019-10-28", "2019-11-01"),

    ("2020-06-15", "2020-06-19"),
    ("2020-11-09", "2020-11-13"),
]

Feature_dataset["swotvac"] = in_date_ranges(
    Feature_dataset.index,
    swotvac_ranges
)


# ============================================================
# 11. EXAM PERIOD
# ============================================================

exam_ranges = [
    ("2016-06-06", "2016-06-24"),
    ("2016-10-31", "2016-11-18"),

    ("2017-06-05", "2017-06-23"),
    ("2017-10-30", "2017-11-17"),

    ("2018-06-04", "2018-06-22"),
    ("2018-10-29", "2018-11-16"),

    ("2019-06-10", "2019-06-28"),
    ("2019-11-04", "2019-11-22"),

    ("2020-06-22", "2020-07-10"),
    ("2020-11-16", "2020-12-04"),
]

Feature_dataset["exam_period"] = in_date_ranges(
    Feature_dataset.index,
    exam_ranges
)


# ============================================================
# 12. REMOTE EXAMS - 2020
# ============================================================

Feature_dataset["remote_exam"] = in_date_ranges(
    Feature_dataset.index,
    [("2020-06-22", "2020-07-10")]
)


# ============================================================
# 13. PUBLIC HOLIDAYS
# ============================================================

public_holidays = pd.to_datetime([

    # 2016
    "2016-01-26",
    "2016-03-25",
    "2016-03-28",
    "2016-03-29",
    "2016-04-25",
    "2016-09-30",
    "2016-12-25",
    "2016-12-26",

    # 2017
    "2017-01-26",
    "2017-04-14",
    "2017-04-17",
    "2017-04-18",
    "2017-04-25",
    "2017-09-29",
    "2017-12-25",
    "2017-12-26",

    # 2018
    "2018-01-01",
    "2018-01-26",
    "2018-03-30",
    "2018-04-02",
    "2018-04-03",
    "2018-04-25",
    "2018-09-28",
    "2018-12-25",
    "2018-12-26",

    # 2019
    "2019-01-01",
    "2019-01-28",
    "2019-04-19",
    "2019-04-22",
    "2019-04-23",
    "2019-04-25",
    "2019-09-27",
    "2019-12-25",
    "2019-12-26",

    # 2020
    "2020-01-01",
    "2020-01-27",
    "2020-04-10",
    "2020-04-13",
    "2020-04-14",
    "2020-04-25",
    "2020-10-23",
    "2020-12-25",
    "2020-12-28",
])

Feature_dataset["public_holiday"] = (
    date.isin(public_holidays)
).astype(int)


# ============================================================
# 14. COVID DISRUPTION
# 9 March 2020 onward
# ============================================================

Feature_dataset["covid_disruption"] = (
    date >= pd.Timestamp("2020-03-09")
).astype(int)


# ============================================================
# 15. ONLINE CLASSES
# 9 - 13 March 2020
# ============================================================

Feature_dataset["online_classes"] = in_date_ranges(
    Feature_dataset.index,
    [("2020-03-09", "2020-03-13")]
)


# ============================================================
# 16. COVID CAMPUS RESTRICTIONS
# 16 March - 18 October 2020
# ============================================================

Feature_dataset["covid_restrictions"] = in_date_ranges(
    Feature_dataset.index,
    [("2020-03-16", "2020-10-18")]
)


# ============================================================
# 17. COVID CURFEW PERIOD
# 2 August - 13 September 2020
# ============================================================

Feature_dataset["covid_curfew_period"] = in_date_ranges(
    Feature_dataset.index,
    [("2020-08-02", "2020-09-13")]
)


# ============================================================
# 18. EFFECTIVE CAMPUS OPEN
#
# Combines:
# - University open
# - Weekday
# - 9am-5pm
# - Not public holiday
# - Not COVID restricted
# ============================================================

Feature_dataset["campus_facility_effectively_open"] = (
    (Feature_dataset["university_open"] == 1) &
    (Feature_dataset["is_weekday"] == 1) &
    campus_time_open &
    (Feature_dataset["public_holiday"] == 0) &
    (Feature_dataset["covid_restrictions"] == 0)
).astype(int)


# ============================================================
# 19. EFFECTIVE LIBRARY OPEN
# ============================================================

Feature_dataset["library_effectively_open"] = (
    (Feature_dataset["library_open_hours"] == 1) &
    (Feature_dataset["university_open"] == 1) &
    (Feature_dataset["public_holiday"] == 0) &
    (Feature_dataset["covid_restrictions"] == 0)
).astype(int)


# ============================================================
# 20. TEACHING DAY
# ============================================================

Feature_dataset["teaching_day"] = (
    (Feature_dataset["semester_active"] == 1) &
    (Feature_dataset["is_weekday"] == 1) &
    (Feature_dataset["mid_sem_break"] == 0) &
    (Feature_dataset["public_holiday"] == 0)
).astype(int)


# ============================================================
# 21. CHECK THE CREATED FEATURES
# ============================================================

feature_columns = [
    "is_weekday",
    "is_weekend",
    "university_open",
    "university_closed",
    "campus_facility_hours",
    "library_open_hours",
    "semester_1",
    "semester_2",
    "semester_active",
    "mid_sem_break",
    "orientation_week",
    "swotvac",
    "exam_period",
    "remote_exam",
    "public_holiday",
    "covid_disruption",
    "online_classes",
    "covid_restrictions",
    "covid_curfew_period",
    "campus_facility_effectively_open",
    "library_effectively_open",
    "teaching_day",
]

print(Feature_dataset[feature_columns].head())
print("\nFeature counts:")
print(Feature_dataset[feature_columns].sum())

                     is_weekday  is_weekend  university_open  \
2016-01-01 00:00:00           1           0                0   
2016-01-01 00:15:00           1           0                0   
2016-01-01 00:30:00           1           0                0   
2016-01-01 00:45:00           1           0                0   
2016-01-01 01:00:00           1           0                0   

                     university_closed  campus_facility_hours  \
2016-01-01 00:00:00                  1                      0   
2016-01-01 00:15:00                  1                      0   
2016-01-01 00:30:00                  1                      0   
2016-01-01 00:45:00                  1                      0   
2016-01-01 01:00:00                  1                      0   

                     library_open_hours  semester_1  semester_2  \
2016-01-01 00:00:00                   0           0           0   
2016-01-01 00:15:00                   0           0           0   
2016-01-01 00:30:00    